In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Silver Orders Validation").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
bronze_orders= spark.read.parquet("../data/bronze/orders")
silver_orders = spark.read.parquet("../data/silver/orders")

In [4]:
print("Bronze count:", bronze_orders.count())
print("Silver count:", silver_orders.count())

Bronze count: 99441
Silver count: 99441


In [5]:
from pyspark.sql.functions import col, when, count

silver_orders.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in silver_orders.columns
]).show(vertical=True)

-RECORD 0-----------------------------
 order_id                      | 0    
 customer_id                   | 0    
 order_status                  | 0    
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



In [6]:
silver_orders.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|995392413cee61cc1fcec9807870ea8c|4bf24904ec428325abad2791fcca1aff|delivered   |2017-09-04 22:24:05     |2017-09-04 22:43:54|2017-09-13 17:20:04         |2017-09-22 21:09:32          |2017-09-27 00:00:00          |
|b39de9ed2bb8fd08a970e4517c698824|ed18b557140ff674f36cbcf73921dbbe|delivered   |2018-03-27 09:15:17     |2018-03-27 09:30:12|2018-03-27 23:5

In [7]:
silver_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [13]:
#approval shouldnt happen before the purchase
invalid_purchase_approval = silver_orders.filter(
    (col("order_approved_at").isNotNull()) &
    (col("order_purchase_timestamp") > col("order_approved_at"))
)
print("Invalid approvals:", invalid_purchase_approval.count())


Invalid approvals: 0


In [14]:
#carrier should not receive the order before customer places the order
invalid_purchase_carrier = silver_orders.filter(
    (col("order_delivered_carrier_date").isNotNull()) &
    (col("order_purchase_timestamp") > col("order_delivered_carrier_date"))
)
print("Invalid carrier dates:", invalid_purchase_carrier.count())

Invalid carrier dates: 166


In [15]:
#The customer cannot receive the package before the carrier has it.
invalid_carrier_delivery = silver_orders.filter(
    (col("order_delivered_carrier_date").isNotNull()) &
    (col("order_delivered_customer_date").isNotNull()) &
    (col("order_delivered_carrier_date") > col("order_delivered_customer_date"))
)

invalid_carrier_delivery.count()

23

In [16]:
#A customer cannot receive an order before purchasing it.
invalid_purchase_delivery = silver_orders.filter(
    (col("order_delivered_customer_date").isNotNull()) &
    (col("order_purchase_timestamp") > col("order_delivered_customer_date"))
)
invalid_purchase_delivery.count()



0